# Experimental Causal Research

**Topic 08 · 2 lectures**

<hr>

# <center><a class="tocSkip"></center>
# <center>HONR 46400 — Evidence-Driven Research</center>
# <center>Professor: Davi Moreira</center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/blob/main/notebooks/student/nb08_experimental_causal_student.ipynb)

---

## 🧭 Inquiry & Claim Boundary

**Inquiry emphasis:** causal reasoning (causal kind). A **causal question** asks
what would change if you intervened, not just what goes with what. Example: "does
a get-out-the-vote call *raise* turnout?", not "do people who were called vote
more?" This week you earn the word *because* the cleanest way there is, by
**randomly assigning** who gets the treatment, so chance, not the kind of person,
decides who is treated.

**Design pathway:** experimental causal. **Experimental** means you control the
assignment: you, not the world, decide who is treated, and you decide it at
random. That single move is what lets a plain difference in outcomes carry a
causal reading.

| | |
|---|---|
| **A claim this topic PERMITS** | "Because assignment was randomized and I checked balance, the difference in means estimates the average treatment effect, reported with a randomization-inference interval and its limitations." |
| **A claim this topic does NOT permit** | "The treated group did better, so the treatment worked," with no check that assignment, not the kind of people in each group, produced the gap; or a clean effect reported while attrition, noncompliance, or spillovers quietly changed which units and which quantity the number describes. |

**Where this sits in the course:** Week 9 of the design library, the experimental
side of the causal question. It builds on the observational-causal identification
work and opens **M8, your minimum viable analysis**, the first real estimate on
your own question, which you pitch at this week's Friday studio.

*Provenance: RDSS ch. 18 'Experimental: causal' + declaration_18.1 (two-arm trial) + rdss foos_etal | potential outcomes, random assignment, ATE, randomization inference, power, and the complications (blocking, clusters, factorial, attrition, noncompliance, spillover) | translated (R→Python) and adapted for an honors non-quant audience with seeded simulations (seed 464)*

## Learning Objectives

By the end of this notebook, you will be able to:

1. State the **potential outcomes** Y(1) and Y(0) behind any causal claim and
   name the fundamental problem that keeps one of them always missing.
2. Explain what **random assignment** buys you, and check **balance** to see
   whether the randomization looks real.
3. Estimate an **average treatment effect** from a real field experiment as a
   difference in means, and put a **randomization-inference** interval around it.
4. Read a **power** curve to find the smallest effect your design could have
   caught, and say when a study is too small to trust.
5. Diagnose how **attrition, noncompliance, and spillovers** each change which
   units and which quantity your estimate honestly describes.
6. Apply potential outcomes, an ATE with its interval, and a named threat to your
   own project's M8 minimum viable analysis.

---

In [ ]:
# Setup — run this cell first.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx  # for the experiment DAG; if missing locally: pip install networkx (pre-installed in Colab)

pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 3)
plt.rcParams["figure.figsize"] = (9, 5)

SEED = 464  # course number — keeps every simulation reproducible
rng = np.random.default_rng(SEED)

# Data loads: GitHub raw URL first (works in Colab), local repo path as fallback.
DATA_URL = ("https://raw.githubusercontent.com/davi-moreira/"
            "2026F_evidence_driven_research_purdue_HONR464/main/notebooks/data/")

def load_course_data(filename):
    """Load a course dataset from GitHub, falling back to the local repo copy."""
    try:
        return pd.read_csv(DATA_URL + filename)
    except Exception:
        from pathlib import Path
        local = Path("notebooks/data") / filename
        if not local.exists():
            local = Path("../data") / filename
        return pd.read_csv(local)

print("✓ Setup complete — seed", SEED)

### 🤝 AI Research Partner

This week you work with **Gemini** as a research partner on the most demanding
claim in the course, a causal one. A good use looks like this: you commit your own
answer first, then ask Gemini to explain a result, propose a check you can run, or
attack a caveat you wrote. A bad use is letting it decide whether your design
identifies an effect, whether an interval is small enough to matter, or which
threat your estimate quietly fell to.

**Never delegate these:** whether random assignment actually happened in your
design, which quantity your number estimates and for whom, and whether the effect
is large enough to act on. Those are yours to declare and defend. The full list of
never-delegate decisions lives in
[`ai_resources/human_responsibility_checklist.md`](https://github.com/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/blob/main/ai_resources/human_responsibility_checklist.md).

**Commit first, then ask.** Every AI block below asks you to write your own answer
before you open Gemini. Commit your work to git before a long AI session, so you
can always compare what you wrote against what the tool changed.

> **A question that often comes up here:** *"A causal claim sounds hard. Can't I
> just let Gemini tell me whether my design proves cause?"* No, and this is the
> one thing to hold onto all week. Gemini cannot see how treatment was assigned in
> your study. It can explain potential outcomes and even write code, but the fact
> that earns *because*, a real randomization, lives in your design, not in the
> model. You confirm it. The tool never does.

---

# Lecture 1

### 🧩 Research Puzzle

*(Your research lead opens the lecture with this. Think it through and commit an
answer before we go further. No AI yet.)*

A campaign ran a get-out-the-vote canvass. Volunteers knocked on some households
and skipped others, and afterward the canvassed households voted at a **higher**
rate than the skipped ones.

Here is the claim on the table: **"canvassing raised turnout."** Would you stake
your name on it, yes or no? Commit to one answer before we go further. Then defend
it: what would have to be true about *who got canvassed* before that turnout gap
is the effect of canvassing rather than the effect of which households the
volunteers chose to visit? Name one specific way volunteers might pick easier or
more-motivated doors, and say which way that would bend the gap. Getting from "the
canvassed voted more" to "canvassing caused it" is the whole job of Lecture 1.

## 1. What an Experiment Actually Estimates

**Guiding question:** *if you could run the world twice, once with the treatment
and once without, what would you compare, and why does randomizing let you fake
that comparison?*

> *"Someone always tells me the treated group did better. Fine. So did the ones
> who were healthier to begin with, or keener, or richer. 'Better after the
> treatment' is not 'the treatment worked.' Show me what the treated group would
> have done without it, or admit you are guessing."*
> — a **skeptical peer** who refuses to confuse *after* with *because*

Start with one unit: one person, one household, one city. The **treatment** is the
thing being tested. Example: a get-out-the-vote call. The **control** condition is
life without it, the comparison world. For that one unit, write two numbers.

- **Potential outcome Y(1)**: the outcome that unit would have *with* the
  treatment. Example: whether this household votes if it is canvassed.
- **Potential outcome Y(0)**: the outcome that *same* unit would have *without* the
  treatment. Example: whether this same household votes if it is left alone.

The causal effect for that unit is the difference, Y(1) minus Y(0): the world with
the treatment minus the world without it. Now the catch that governs the whole
field, the **fundamental problem of causal inference**: for any one unit you only
ever see *one* of those two numbers. A household was either canvassed or not. You
never observe both its Y(1) and its Y(0). No amount of staring at what a unit
actually did recovers the outcome it *would* have had the other way.

You gave up on the individual effect. Here is what you chase instead. The
**average treatment effect (ATE)** is the average of Y(1) minus Y(0) across all
your units, the typical effect rather than any one unit's. Example: on average
across households, canvassing raised the chance of voting by a few points. You
cannot see any single unit's effect, but the average has an honest estimate, and
one move unlocks it.

That move is **random assignment**: letting pure chance, a coin flip or a lottery,
decide which units get the treatment. Example: a computer flips a fair coin for
each household to decide canvass-or-skip. Chance knows nothing about motivation,
age, or income, so the treated and control groups it builds are alike on average
in every respect, measured and unmeasured. The control group's average outcome
then honestly stands in for what the treated group *would* have done untreated.

One villain is what randomization defeats. A **confounder** is a third trait that
pushes on both who gets treated and the outcome. Example: motivated voters are
both easier to reach at home *and* more likely to vote anyway, so canvassed
households look better even if the knock did nothing. Confounders are exactly why
"the treated did better" proves so little on its own. Random assignment breaks the
link between the confounder and who gets treated, so no trait can steer who lands
where. The picture below shows that break.

In [ ]:
# The experiment DAG: what randomization severs.
# (networkx is imported once in the setup cell.)
G = nx.DiGraph()
pos = {
    "Random\nassignment Z": (0.0, 0.0),
    "Outcome Y":            (2.4, 0.0),
    "Hidden traits U\n(motivation, age, ...)": (1.2, 1.4),
}
G.add_edges_from([
    ("Random\nassignment Z", "Outcome Y"),
    ("Hidden traits U\n(motivation, age, ...)", "Outcome Y"),
])

fig, ax = plt.subplots(figsize=(9, 4.6))
nx.draw_networkx_nodes(G, pos, node_color="#DCE6F1", edgecolors="#4C72B0",
                       linewidths=1.5, node_size=7000, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=9, ax=ax)
nx.draw_networkx_edges(G, pos, arrowstyle="-|>", arrowsize=18, edge_color="#555555",
                       width=1.6, node_size=7000, ax=ax)
# The arrow that randomization DELETES: U -> Z. Draw it crossed out, in gray.
ax.annotate("", xy=(0.35, 0.35), xytext=(1.0, 1.15),
            arrowprops=dict(arrowstyle="-|>", color="#999999", ls="--", lw=1.4))
ax.text(0.30, 0.95, "randomization\ndeletes this arrow\n(U to Z)", fontsize=8,
        color="#555555", ha="center",
        bbox=dict(boxstyle="round", fc="white", ec="#999999"))
ax.set_title("A randomized experiment: chance cuts the link from hidden traits to treatment")
ax.axis("off")
ax.set_xlim(-0.7, 3.5)
ax.set_ylim(-0.6, 2.0)
plt.tight_layout()
plt.show()

print("✓ Experiment DAG drawn.")
print("  Z (assignment) and U (hidden traits) both feed Y, but because Z is random,")
print("  U cannot feed Z. That deleted arrow is what makes the comparison causal.")

**Reading the diagram.** Two things affect the outcome Y: the treatment Z and a
cloud of hidden traits U you never measured. In an observational study, U also
steers who gets treated, and that back path is the confounding you cannot untangle.
Random assignment deletes the arrow from U to Z. Chance, not motivation or income,
now decides treatment, so the treated and control groups differ only by the coin
flip. That deleted arrow is the entire reason an experiment can say *because*.

*In plain terms, the picture says: hidden differences still affect the outcome,
but randomizing makes sure they no longer decide who gets treated.*

### 🔮 Pause & Predict

The cell below builds a tiny **toy world** of six units where, unlike real life,
you get to see *both* Y(1) and Y(0) for everyone, so you can read the true average
treatment effect straight off the table. Before you run it, commit: with only six
units, once each shows just one of its two potential outcomes, do you think a
single random assignment will land close to the true ATE, or could it miss badly,
even come out the wrong sign? Write one sentence of reasoning.

### YOUR ANSWER HERE:

**Close to the true ATE, or could it miss badly, and why:**

---

In [ ]:
# A TOY world (illustration, not data): the science table you never fully see.
# Y0 and Y1 are both shown here ONLY because we built the world; in real life you
# see one column per unit. Everything downstream is derived from these two arrays.
Y0 = np.array([3, 5, 4, 6, 2, 5])   # outcome each unit would show WITHOUT treatment
Y1 = np.array([6, 7, 5, 9, 4, 6])   # outcome the SAME unit would show WITH treatment
unit_effect = Y1 - Y0               # the causal effect for each unit
true_ATE = unit_effect.mean()

science = pd.DataFrame({"unit": range(1, 7), "Y0": Y0, "Y1": Y1, "effect Y1-Y0": unit_effect})
print("The full science table (both potential outcomes, visible only because we built it):")
print(science.to_string(index=False))
print(f"\nTrue average treatment effect (mean of Y1 - Y0) = {true_ATE:.2f}")

# One randomized experiment: assign 3 of 6 to treatment, observe ONE column each.
rng_toy = np.random.default_rng(SEED)
z = np.zeros(6, dtype=int)
z[rng_toy.permutation(6)[:3]] = 1
observed = np.where(z == 1, Y1, Y0)      # each unit reveals only its assigned world
est_ATE = observed[z == 1].mean() - observed[z == 0].mean()
print(f"\nOne random assignment z = {z.tolist()}")
print(f"Difference in means from this one experiment = {est_ATE:.2f}   "
      f"(true ATE = {true_ATE:.2f})")

**Reading the evidence.** The true ATE is exactly 2.00, because you can average
the six individual effects you were allowed to see. A real experiment never gets
that table. It sees one column per unit, treated units showing Y(1) and control
units showing Y(0), and estimates the ATE as the difference in group averages.
This one six-unit draw came out around -0.3, nowhere near the true +2.00 and even
the wrong sign, because six coin flips are far too lumpy to trust. That is the
warning to carry: a single tiny experiment can land anywhere. Bigger experiments
smooth that lumpiness, which is the whole reason sample size matters below.

> **A question that often comes up here:** *"If we can never see both outcomes for
> one unit, isn't causal inference just impossible?"* The individual effect really
> is unrecoverable, and that part is impossible. What random assignment rescues is
> the **average** effect, because chance makes the two groups interchangeable. You
> trade "what did the treatment do to *this* unit?" for "what did it do *on
> average*?", and the second question has an honest answer.

---

## 2. A Real Experiment, Analyzed Honestly

**Guiding question:** *when chance assigned who got contacted, what exactly does
the difference in turnout estimate, and how sure can you be it is not luck?*

The `foos_etal` file is a **real** UK get-out-the-vote field experiment. Chance
assigned some voters to a contact arm (the treatment) and left others as the
control arm, and whether each voter turned out in 2014 was recorded afterward. You
will analyze it with the simplest honest tool, the **difference in means** between
the two arms, and be strict about two disciplines.

- Say *exactly* what you compute: an **unweighted, simple difference in means**
  across the two arms.
- Say *exactly* what you do not. The original study ran a more elaborate
  **weighted** analysis (the `weights` column is the clue). Your number reproduces
  the experiment's *logic*, randomize then compare arms. It does not reproduce the
  paper's headline estimate, and claiming it did would be the failure.

### 🔮 Pause & Predict

Before running the next cell, commit to two predictions. Did being contacted
*raise* turnout? If so, by roughly how many percentage points? Write a direction
and a number. You are predicting a real result, so hold your guess even if the
data surprises you.

### YOUR ANSWER HERE:

**Did contact raise turnout? (up / down / no change):**

**By roughly how many percentage points:**

**Why:**

---

### 🛠️ Run the Study: the canvass experiment, end to end

Run the next cell. It loads the experiment, checks **balance** (whether the arms
look alike before treatment), and computes the difference in turnout with a
standard error and a 95% interval. Read the printout before the next markdown cell.

**🔴 Live in class: we run this one together.**
**Before you ask:** write one sentence predicting whether the 95% interval will
exclude zero (you gave a size last cell; now commit to whether it clears zero).

> 💡 **Gemini Prompt:** "Here is Python that analyzes a real get-out-the-vote
> experiment as a simple unweighted difference in turnout between a contacted arm
> and a control arm, with a two-proportion standard error and a 95% interval:
> [paste the next cell]. Explain, line by line, what each quantity is, and why
> randomization is what lets a plain difference in means carry a causal reading.
> Then give me one independent way to confirm the difference I printed, without
> rerunning your own code."
>
> **After running, verify (counters *confident fabrication*: a fluent line-by-line
> walkthrough can describe a result the code never produced):**
> - [ ] Confirm the difference, standard error, and interval Gemini quotes match
>   the exact numbers your cell printed, not values it guessed. If it says the
>   interval is symmetric and yours is not, trust your printout.
> - [ ] Run its "independent way to confirm" yourself (a hand recomputation of the
>   two turnout rates and their gap). If it will not reproduce your number, keep yours.
> - [ ] Log this use in your AI Research Ledger: task, tool, decision, verification.

In [ ]:
# Load the real experiment and confirm its shape before trusting anything.
foos = load_course_data("foos_etal.csv")
assert foos.shape == (8375, 5), "unexpected shape — flag this!"
print("✓ Loaded foos_etal.csv —", foos.shape[0], "rows,", foos.shape[1], "columns")
print()

# Turnout by arm (UNWEIGHTED, simple).
arms = foos.groupby("treat")["marked_register_2014"].agg(n="count", turnout="mean")
arms.index = ["control (treat=0)", "contacted (treat=1)"]
print("Turnout by arm:")
print(arms.round(4))
print()

# Balance check: do the arms look alike BEFORE treatment?
w_by_arm = foos.groupby("treat")["weights"].mean()
ward_share = pd.crosstab(foos["ward"], foos["treat"], normalize="columns")
ward_gap = (ward_share[1] - ward_share[0]).abs().max()
print(f"Balance — mean design weight: control {w_by_arm[0]:.2f}, contacted {w_by_arm[1]:.2f}")
print(f"Balance — largest gap in any ward's share between arms: {ward_gap:.3f}")
print()

# The average treatment effect as a difference in means + its 95% interval.
t1 = foos.loc[foos.treat == 1, "marked_register_2014"]
t0 = foos.loc[foos.treat == 0, "marked_register_2014"]
diff = t1.mean() - t0.mean()
se = np.sqrt(t1.mean() * (1 - t1.mean()) / len(t1)
             + t0.mean() * (1 - t0.mean()) / len(t0))
ci_lo, ci_hi = diff - 1.96 * se, diff + 1.96 * se
print(f"ATE (contacted - control): {diff*100:+.1f} percentage points")
print(f"standard error: {se*100:.1f} points")
print(f"95% interval: [{ci_lo*100:+.1f}, {ci_hi*100:+.1f}] points   (excludes 0: {ci_lo > 0})")

In [ ]:
# Self-check: arm sizes, direction, and interval are what we reported.
assert len(t1) == 6104 and len(t0) == 2271, "arm sizes changed — investigate"
assert diff > 0, "our estimate says contact RAISED turnout — report the sign honestly either way"
assert ci_lo > 0, "the 95% interval excludes zero"
assert w_by_arm[0] > 2 * w_by_arm[1], "the arms differ sharply on design weights (unequal assignment probabilities)"
print(f"✓ Self-check passed: contacted arm n={len(t1)}, control arm n={len(t0)}.")
print(f"✓ Unweighted ATE = {diff*100:+.1f} points, interval [{ci_lo*100:+.1f}, {ci_hi*100:+.1f}] points.")
print("✓ Wards line up closely, but weights differ a lot by arm — the design used unequal")
print("  assignment probabilities, so this unweighted number is the classroom-simple version.")

Two things stand out. The arms line up on wards (largest gap under a few
percentage points), which is what balanced randomization should look like. The
**weights** do not line up: the control arm carries much heavier weights than the
contacted arm. That is the fingerprint of a design that assigned treatment with
**unequal probabilities** and corrects for it with weights. Your unweighted
difference ignores that correction, which is exactly why it reproduces the
experiment's logic rather than the paper's exact number.

Now put uncertainty around the estimate the way an experiment earns it.
**Randomization inference** asks a blunt question: if the treatment truly did
nothing, how often would pure chance alone hand me a gap this big? You answer it by
**re-randomizing**: shuffle the treatment labels thousands of times as if redoing
the coin flips, recompute the difference each time, and build the **sharp-null
reference distribution**, the spread of fake gaps you would see under no effect.
Example: shuffle who was "contacted" 2,000 times and see how many shuffles beat
your real gap. If almost none do, chance is a poor explanation for your result.

### 🛠️ Hands-On: put an interval on the effect by re-randomizing

Run the cell. It shuffles the treatment labels 2,000 times under the sharp null of
no effect, builds the reference distribution, marks your real difference against
it, and reports a randomization-inference interval and p-value. Read the picture
before the next markdown cell.

In [ ]:
# Randomization inference: re-randomize under the sharp null of NO effect.
# Fresh seeded generator so this result is reproducible on its own (seed 464).
rng_ri = np.random.default_rng(SEED)
y = foos["marked_register_2014"].to_numpy()
n_treat = int(foos["treat"].sum())

B = 2000
null = np.empty(B)
for b in range(B):
    perm = rng_ri.permutation(len(y))
    mask = np.zeros(len(y), dtype=bool)
    mask[perm[:n_treat]] = True          # a fresh random "treated" set of the same size
    null[b] = y[mask].mean() - y[~mask].mean()

sd_null = null.std(ddof=1)               # the randomization standard error
ri_lo, ri_hi = diff - 1.96 * sd_null, diff + 1.96 * sd_null
p_two = np.mean(np.abs(null) >= abs(diff))
n_as_extreme = int((np.abs(null) >= abs(diff)).sum())

print(f"Observed ATE                     : {diff*100:+.1f} points")
print(f"Randomization standard error     : {sd_null*100:.2f} points")
print(f"Randomization-inference interval : [{ri_lo*100:+.1f}, {ri_hi*100:+.1f}] points")
print(f"Two-sided p-value                : {p_two:.4f}  "
      f"({n_as_extreme} of {B} reshuffles matched or beat the real gap)")

fig, ax = plt.subplots()
ax.hist(null * 100, bins=40, color="#4C72B0", edgecolor="white",
        label="fake gaps under NO effect (2,000 reshuffles)")
ax.axvline(diff * 100, color="#C44E52", linestyle="--", linewidth=2,
           label=f"observed gap = {diff*100:+.1f} pts")
ax.axvline(0, color="gray", linestyle=":", linewidth=1.2, label="no effect (0)")
ax.set_xlabel("Difference in turnout between arms (percentage points)")
ax.set_ylabel("Number of reshuffles")
ax.set_title("Randomization inference: the real gap sits in the far tail of the no-effect world")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Self-check: the randomization SE matches the analytic one, and the effect is rare under the null.
assert 0.010 < sd_null < 0.015, "randomization SE drifted — is the seed 464 and B=2000?"
assert ri_lo > 0, "the randomization-inference interval should exclude zero"
assert p_two < 0.02, "the observed gap should be rare under the sharp null"
print(f"✓ Self-check passed: randomization SE ({sd_null*100:.2f} pts) matches the analytic SE "
      f"({se*100:.2f} pts).")
print(f"✓ Only {n_as_extreme} of {B} no-effect reshuffles reached the real gap, so chance is a "
      f"poor explanation (p = {p_two:.3f}).")

### 🔍 Reading the Evidence: write the identification argument

The reference distribution centers on zero (no effect) and your real gap sits out
in its far tail, so pure chance is an unconvincing explanation. In the cell below,
write the **three-sentence identification argument** for this experiment. First:
the quantity you want, the average effect of contact on turnout. Second: why the
control arm is a valid stand-in for what the contacted arm would have done
untreated. Third: what therefore follows about the difference in means. Then add a
fourth sentence naming the one caveat your *unweighted* pass carries.

### YOUR ANSWER HERE:

**(1) The quantity I want:**

**(2) Why the control arm is a valid counterfactual:**

**(3) Therefore:**

**(4) The caveat my unweighted analysis carries:**

---

---

### ⏸ In class through here

**You have reached the end of Lecture 1's in-class block.** Everything below
deepens the same ideas. Finish it on your own before the next lecture: the design
choice, the power simulation and its homework-depth prompt, and the
potential-outcomes practice. Come back with the first computed estimate on your own
project's branch started for M8.

---

### ⚖️ Make a Design Choice: what does your number estimate?

*(Human-first: settle your own choice and its defense before you ask any AI.)*

You want to test whether a short study-skills workshop raises exam scores. You have
40 volunteers. **Choose one design and defend it** in a short paragraph, naming
what its number would estimate and what it could not rule out.

- **A.** Let the 40 sign up for the workshop if they want, then compare workshop-goers'
  scores to non-goers'.
- **B.** Randomly assign 20 of the 40 to the workshop and 20 to a control, then
  compare the two groups' scores.
- **C.** Give all 40 the workshop and compare their scores to last year's class.

In the cell below, pick A, B, or C, then defend it: what does your chosen design
let you claim about the *effect* of the workshop, and where must that claim stop?

### YOUR ANSWER HERE:

**My chosen design (A / B / C):**

**What its number estimates, and what it cannot rule out:**

**What I can claim about the effect, and where it must stop:**

---

### Power: the smallest effect your design could catch

One more idea decides whether an experiment is worth running. **Statistical power**
is the chance your study would detect a real effect of a given size if one exists.
Example: a design with 30 percent power to catch a 5-point effect would miss that
real effect seven times out of ten. Its partner is the **minimum detectable effect
(MDE)**, the smallest true effect your design has a good chance (usually 80
percent) of catching. Example: with a certain sample size, you might only reliably
catch effects of 14 points or larger, so a real 4-point effect would slip past
unseen. Power rises with sample size, which is why a tiny study can fail to find an
effect that is genuinely there.

### 🔮 Pause & Predict

The next cell simulates many two-arm trials at two sample sizes, 50 per arm and 200
per arm, and reports how often each detects effects of various sizes. Before you
run it, commit: at 50 per arm, what is the largest effect (in points) you think the
study will *still* miss most of the time? Write a number and one sentence of why.

### YOUR ANSWER HERE:

**Largest effect 50-per-arm still misses most of the time, and why:**

---

**🏠 Homework depth — run this on your own before the next lecture.**
**Before you ask:** in one sentence, predict whether doubling the sample size roughly
doubles the power to catch a small effect, or something else. Commit before asking.

> 💡 **Gemini Prompt:** "This cell simulates two-arm trials at two sample sizes and
> estimates statistical power, the chance of detecting a true effect of each size:
> [paste the next cell]. Confirm what 'power' and 'minimum detectable effect' mean
> here, then tell me one reason a real study might report 'no significant effect'
> even when a true effect of a few points genuinely exists. Give me the concrete
> assumption your reasoning depends on."
>
> **After running, verify (counters *plausible-but-wrong-method*: 'not significant'
> is being sold as 'no effect,' which the power table refutes):**
> - [ ] Read the printed power for a small effect at 50 per arm off your own output.
>   If it is low, then "no significant effect" here means *underpowered*, not "no
>   effect." Reject any Gemini sentence that equates the two.
> - [ ] Check its claim about doubling the sample against the two rows you printed:
>   did power to catch a small effect actually rise the way it says?
> - [ ] Log this use in your AI Research Ledger: task, tool, decision, verification.

In [ ]:
# Power simulation: how often does a two-arm trial DETECT a true effect?
# Fresh seeded generator (seed 464); binary outcome with a control rate near foos.
rng_pow = np.random.default_rng(SEED)
p0 = 0.47                              # control-arm success rate
taus = [0.03, 0.05, 0.08, 0.10, 0.14, 0.18]   # true effect sizes to test

def power_at(n_per_arm, tau, reps=800):
    """Share of simulated trials whose 95% test detects the effect."""
    p1 = p0 + tau
    hits = 0
    for _ in range(reps):
        c = rng_pow.random(n_per_arm) < p0
        t = rng_pow.random(n_per_arm) < p1
        pooled = (c.sum() + t.sum()) / (2 * n_per_arm)
        se_p = np.sqrt(pooled * (1 - pooled) * (2 / n_per_arm))
        if se_p == 0:
            continue
        if abs((t.mean() - c.mean()) / se_p) > 1.96:
            hits += 1
    return hits / reps

power_table = pd.DataFrame(
    {f"n={n}/arm": [power_at(n, tau) for tau in taus] for n in [50, 200]},
    index=[f"{int(tau*100)} pts" for tau in taus])
power_table.index.name = "true effect"
print("Power (chance of detecting the effect) by true effect size and sample size:")
print(power_table.round(2).to_string())

fig, ax = plt.subplots()
x = [int(tau * 100) for tau in taus]
ax.plot(x, power_table["n=50/arm"], "o-", color="#4C72B0", label="50 per arm")
ax.plot(x, power_table["n=200/arm"], "s-", color="#55A868", label="200 per arm")
ax.axhline(0.8, color="#C44E52", linestyle="--", linewidth=1.5, label="80% power target")
ax.set_xlabel("True effect size (percentage points)")
ax.set_ylabel("Power (share of trials that detect it)")
ax.set_title("A small study misses real effects: power rises with effect size and sample size")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Self-check: small studies are underpowered for small effects.
assert power_table.loc["5 pts", "n=50/arm"] < 0.25, "50/arm should barely detect a 5-pt effect"
assert power_table.loc["14 pts", "n=200/arm"] > 0.7, "200/arm should usually catch a 14-pt effect"
mde200 = next(int(tau*100) for tau in taus if power_at(200, tau) >= 0.8)
print(f"✓ Self-check passed: at 50 per arm, a 5-point effect is detected only "
      f"{power_table.loc['5 pts','n=50/arm']:.0%} of the time.")
print(f"✓ At 200 per arm, the minimum detectable effect (80% power) is about {mde200} points.")
print("  A real 3-4 point effect, like the canvass result, would slip past a small study unseen.")

**Reading the evidence.** The table tells a sobering story. At 50 per arm, even a
sizable effect is missed most of the time, and a small 3-to-5 point effect is
almost invisible. At 200 per arm, you reliably catch larger effects but still need
roughly a 14-point true effect for 80 percent power. The canvass effect you found
was about 3.4 points, and that experiment had thousands of voters for a reason. The
lesson to carry: a study that reports "no significant effect" may simply have been
too small to see the effect it was looking for, and power is how you tell the two
apart before you run.

> **A question that often comes up here:** *"I got a p-value below 0.05 from just
> twelve participants. Isn't that proof the effect is real?"* Be careful. A tiny
> study that does clear the bar tends to do so only when noise happens to line up,
> so its estimate is usually inflated, and it will not replicate. That is why a
> randomization-inference interval, which shows the whole range of effects
> compatible with your data, tells you more than a lone p-value. With twelve
> people, that interval is enormous, and honesty means reporting its width, not
> just that it cleared zero.

### 📝 Practice: name the potential outcomes, then judge the power

*(Human-first: work all three yourself before any AI. The reasoning is the graded
skill.)*

For each study, write **Y(1) and Y(0) for one named unit**, mark **which one is
actually observed**, and say whether the study is powered to detect a **small,
medium, or only a large** effect. Answer in the cell that follows.

- **A.** A campus club randomly gives 800 first-years a text-message reminder to
  register to vote; 800 others get nothing. Outcome: whether each registers.
- **B.** A lab tests a focus app on 8 volunteers, each measured with and without the
  app on two different days.
- **C.** A city randomly assigns 30 of 60 neighborhoods to get new streetlights and
  compares reported nighttime crashes a year later.

### YOUR ANSWER HERE:

**A (Y(1) / Y(0) for one unit / which is observed / power):**

**B (Y(1) / Y(0) for one unit / which is observed / power):**

**C (Y(1) / Y(0) for one unit / which is observed / power):**

---

You built potential outcomes, estimated a real ATE, and put a
randomization-inference interval around it. Lecture 2 turns to what breaks a clean
experiment in the real world: units drop out, assigned people do not comply, and
neighbors talk to each other. Each one changes which quantity your number
describes, and naming that is the heart of your M8.

---

---

# Lecture 2

### 🧩 Research Puzzle

*(Your research lead opens the lecture with this. Think it through and commit an
answer before we go further. No AI yet.)*

You ran a clean randomized experiment. The treatment "worked": the treated group's
outcome came out ahead. Then you notice something. **A third of the participants
dropped out before you measured the outcome**, and they dropped out more from one
arm than the other.

Here is the question on the table: **what did your number actually estimate, and
for whom?** Commit an answer before we go further. If the people who quit were
different from the people who stayed, and quit at different rates by arm, is your
surviving-group difference still the effect for *everyone you started with*? Name
one reason a person might drop out that is tied to how well the treatment was
working for them, and say which way that would bend your estimate. Pinning down
*which units and which quantity* your number describes is the whole job of Lecture
2.

## 3. When a Clean Experiment Breaks

**Guiding question:** *your randomization was perfect on paper, then people dropped
out and neighbors talked to each other. What did you actually estimate?*

> *"A clean randomization is a promise made at the start. What I check is whether
> the world kept it. People vanish before you measure them, assigned people ignore
> the treatment, and treated neighbors leak the effect onto controls. Any one of
> those changes the sentence you are allowed to write."*
> — an **IRB reviewer** who has watched tidy designs meet messy reality

Randomization protects you at the moment of assignment. Three common problems
strike *after* that moment, and each one quietly changes which quantity your
difference in means estimates.

- **Attrition**: units that leave the study before you measure the outcome.
  Example: sick patients drop out of the arm where the drug had side effects.
  When who leaves is tied to the outcome, the survivors are no longer a fair
  comparison.
- **Noncompliance**: assigned units that do not take the treatment they were
  offered (or controls that get it anyway). Example: half the people mailed a
  voter guide never open it. Your assignment was random, but who actually
  *received* treatment is not.
- **Spillover** (also called **interference**): treatment reaching control units
  indirectly. Example: a canvassed voter reminds an untreated neighbor to vote, so
  the control group is partly treated too. That lifts the control outcome and
  shrinks the gap you measure.

### 🔮 Pause & Predict

The next cell builds a clean two-arm world with a **true effect of 4 points**,
confirms a plain analysis recovers it, then lets **low-outcome treated units drop
out** more often, the attrition that is tied to the outcome. Before you run it,
commit: when those struggling treated units disappear, will the survivors-only
estimate drift *up* or *down* from the true 4 points? Write the direction and one
sentence of why.

### YOUR ANSWER HERE:

**Up or down, and why:**

---

### 🛠️ Hands-On: inject attrition, then spillover

Run the two cells below. The first builds the clean world, then applies
outcome-related attrition and reports the survivors-only estimate. The second takes
a fresh clean world and lets treated units leak the effect onto their control
neighbors. Watch the estimate move away from the true 4 points each time.

**🔴 Live in class: we run this one together.**
**Before you ask:** write one sentence naming, for attrition, *which group* the
survivors-only number now describes (hint: it is not everyone you started with).

> 💡 **Gemini Prompt:** "This cell starts from a randomized experiment with a known
> true effect of 4 points, then removes low-outcome units from the treated arm and
> recomputes the difference in means: [paste the attrition cell]. State precisely
> which population the survivors-only estimate now describes, and why that is a
> different quantity from the effect for everyone originally randomized. Do not just
> say 'it is biased'; name the group and the direction."
>
> **After running, verify (counters *silent scope change*: 'the estimate is biased'
> quietly swaps the real issue, which is that the number now answers a different
> question about a different group):**
> - [ ] Check that Gemini's stated direction matches your printout: the survivors-only
>   estimate came out *above* the true 4 points, so reject any claim that it runs low.
> - [ ] Make it name the group. "The effect among treated units who did not drop out"
>   is a real answer; "it is biased" is not. If it will not name the group, push again.
> - [ ] Log this use in your AI Research Ledger: task, tool, decision, verification.

In [ ]:
# ATTRITION: a clean world with a TRUE effect of 4, then outcome-related dropout.
# Fresh seeded generator (seed 464). Everything downstream derives from Y0, z, stay.
rng_attr = np.random.default_rng(SEED)
tau = 4.0
n = 1200                                     # 600 per arm
Y0 = rng_attr.normal(50, 10, n)              # outcome without treatment
Y1 = Y0 + tau                                # constant true effect of 4 points
z = np.zeros(n, dtype=int)
z[rng_attr.permutation(n)[:600]] = 1         # random assignment, 600 treated
Yobs = np.where(z == 1, Y1, Y0)

clean_est = Yobs[z == 1].mean() - Yobs[z == 0].mean()   # plain analysis recovers ~4

# Now attrition tied to the outcome: struggling (low-Yobs) treated units drop out more.
stay_p = np.where(z == 1, 1 / (1 + np.exp(-(Yobs - 48) / 6)), 0.92)
stay = rng_attr.random(n) < stay_p
survivor_est = Yobs[(z == 1) & stay].mean() - Yobs[(z == 0) & stay].mean()

print(f"True effect built into the world      : {tau:+.1f} points")
print(f"Clean analysis (everyone)             : {clean_est:+.2f} points   (recovers the truth)")
print(f"Treated-arm retention                 : {stay[z==1].mean():.0%}  "
      f"(overall {stay.mean():.0%})")
print(f"Survivors-only estimate               : {survivor_est:+.2f} points   "
      f"(off by {survivor_est - tau:+.2f})")

In [ ]:
# Self-check: attrition tied to the outcome inflates the survivors-only estimate.
assert abs(clean_est - tau) < 1.0, "the clean analysis should land near the true effect of 4"
assert survivor_est > tau + 1.5, "outcome-related attrition should push the estimate well above 4"
assert stay[z == 1].mean() < 0.8, "treated-arm retention should be clearly reduced"
print(f"✓ Self-check passed: the clean estimate ({clean_est:+.2f}) recovers the true +4, but "
      f"dropping low-outcome\n  treated units lifts the survivors-only estimate to "
      f"{survivor_est:+.2f}. Same experiment, different quantity.")

In [ ]:
# SPILLOVER: a fresh clean world where treated units leak the effect onto neighbors.
# Fresh seeded generator (seed 464). Units live in clusters; controls near many
# treated neighbors get a partial dose, contaminating the comparison group.
rng_sp = np.random.default_rng(SEED)
n = 800                                      # 400 per arm
Y0 = rng_sp.normal(50, 10, n)
clusters = rng_sp.integers(0, 80, n)         # 80 neighborhoods
z = np.zeros(n, dtype=int)
z[rng_sp.permutation(n)[:400]] = 1
frac_treated_nbrs = np.array([z[clusters == clusters[i]].mean() for i in range(n)])

Y1 = Y0 + tau
# Control units get lifted by the share of treated neighbors in their cluster.
Yobs = np.where(z == 1, Y1, Y0) + (z == 0) * 6.0 * frac_treated_nbrs
spill_est = Yobs[z == 1].mean() - Yobs[z == 0].mean()

print(f"True effect built into the world : {tau:+.1f} points")
print(f"Estimate WITH spillover          : {spill_est:+.2f} points   "
      f"(off by {spill_est - tau:+.2f})")
print("Spillover lifted the CONTROL outcome, so the gap you measure shrinks below the truth.")

In [ ]:
# Self-check: spillover onto controls attenuates the measured effect.
assert spill_est < tau - 1.0, "spillover onto controls should shrink the estimate below 4"
assert spill_est > 0, "the effect should still read positive, just too small"
print(f"✓ Self-check passed: with treated-to-control spillover, the measured effect falls to "
      f"{spill_est:+.2f}\n  against a true +4. The control group was partly treated, so the "
      f"comparison understates the effect.")

### 🔍 Reading the Evidence: which units, which quantity

Both threats left the randomization untouched and still moved the number, in
*opposite* directions. Attrition pushed the estimate **up** (to about +8) because
the survivors were a selected group; spillover pushed it **down** (to about +1.6)
because the control group was partly treated. In the cell below, write three
things. First: for the attrition result, the exact group your survivors-only
estimate describes, and the honest sentence you could report about it. Second: for
spillover, why the true effect is *larger* than what you measured. Third: name the
one design or measurement fix you would add to your own project to guard against
whichever of these two threats is nearer your design.

### YOUR ANSWER HERE:

**Attrition: the group my survivors-only estimate describes, and the honest sentence:**

**Spillover: why the true effect is larger than the measured one:**

**The fix nearest my own project:**

---

---

### ⏸ In class through here

**You have reached the end of Lecture 2's in-class block.** Everything below feeds
your M8 minimum viable analysis directly. Finish it before Friday's studio: adapt
the threat prompt to your own design, interrogate what comes back, work the
noncompliance and design-variant cells, then make the AI-free ethics-and-scope
decision and draft your M8 spine. The 🧑‍⚖️ Human-Only Checkpoint is the
never-delegate core of M8, so give it real time.

---

### 🔁 Modify the Prompt

Above, you had Gemini name which group and quantity the *attrition* estimate
describes. Now do it for your own project. First, in one sentence, name the single
threat nearest your own design: attrition, noncompliance, or spillover. Do not let
AI pick it. Then adapt the prompt so Gemini pressure-tests *your* estimate against
*that* threat.

> **Base prompt (from above):** "This cell starts from a randomized experiment with
> a known true effect, then [applies a threat] and recomputes the difference. State
> precisely which population the estimate now describes, and the direction of the
> change. Do not just say 'it is biased'; name the group and the direction."

Rewrite it so it names *your* treatment, *your* outcome, and the threat you fear
most. In the cell below: name your nearest threat, paste your adapted prompt,
predict the group and direction Gemini will name, then run it and note whether your
prediction held.

### YOUR ANSWER HERE:

**My nearest threat (attrition / noncompliance / spillover) and why:**

**My adapted prompt (my treatment, my outcome, my threat):**

**The group and direction I predict Gemini will name:**

**What Gemini actually named, and how it changed my M8 plan:**

---

### 🔬 Interrogate the Output

Gemini just told you which group your threatened estimate describes. Do not accept
it on faith. Interrogate the answer against the numbers, using four checks. Answer
each in the cell below.

- **Claims:** does the direction it named match your printout? Your attrition
  estimate came out *above* the true effect and your spillover estimate *below* it,
  so an answer that flips either direction is simply wrong.
- **Assumptions:** what does the answer assume about *why* units left or leaked? Is
  that mechanism one you built, or one the tool invented for you?
- **Missing information:** which real weakness did it skip? An answer that names the
  bias but never states the *size* of the drift, or the retention rate, left the
  most useful number on the table.
- **Overstatements:** does any sentence sound more certain than a single simulated
  draw supports? Flag the exact words.

And the rule that governs the whole notebook: **code running without errors is not
the same as code being correct.** A cell can execute cleanly and still compute the
wrong quantity. Verify the *number*, not the green check.

### YOUR ANSWER HERE:

**Claims (does the direction match my numbers?):**

**Assumptions (invented mechanism?):**

**Missing information (the number it skipped):**

**Overstatements (exact words):**

---

### Noncompliance: the effect you can still recover

Attrition and spillover bend the estimate. **Noncompliance** does something
subtler. When assigned-to-treat units do not take the treatment, the plain
assigned-groups comparison, called the **intent-to-treat (ITT)** effect (the effect
of *being offered* the treatment, whether or not you took it), gets diluted toward
zero. Example: if only 60 percent of those offered a workshop attend, the ITT
averages the attendees' gains with the non-attendees' zeros. There is an honest
repair. The **complier average causal effect (CACE)**, sometimes called the local
effect, is the effect *among the units that actually comply*, and under clean
assumptions it equals the ITT divided by the compliance rate. Example: an ITT of
about 2.9 points at 62 percent compliance implies a complier effect near 4.7
points.

**🏠 Homework depth — run this on your own.**
**Before you ask:** in one sentence, predict what the ITT will be relative to the
true complier effect of 5, and why. Commit before asking.

> 💡 **Gemini Prompt:** "This cell simulates a trial where only some of the assigned
> arm actually take the treatment, then computes the intent-to-treat effect and
> divides it by the compliance rate to recover the complier effect: [paste the
> noncompliance cell]. Confirm what intent-to-treat and complier-effect each mean,
> and name the one assumption that has to hold for ITT-divided-by-compliance to be a
> valid complier effect rather than a misleading number."
>
> **After running, verify (counters *plausible-but-wrong-method*: dividing by
> compliance is only valid under an assumption the tool may skip):**
> - [ ] Check the printed ITT, compliance rate, and recovered effect. Confirm the
>   recovered effect equals ITT divided by compliance, using your actual numbers.
> - [ ] Make Gemini state the assumption (no defiers, and assignment affects the
>   outcome only through take-up). If it presents the division as always valid,
>   that is the wrong-method failure; reject it.
> - [ ] Log this use in your AI Research Ledger: task, tool, decision, verification.

In [ ]:
# NONCOMPLIANCE: only 60% of the assigned-treat arm actually take the treatment.
# Fresh seeded generator (seed 464). True effect exists ONLY for those who take it.
rng_nc = np.random.default_rng(SEED)
n = 4000                                      # 2000 per arm
Y0 = rng_nc.normal(50, 10, n)
z = np.zeros(n, dtype=int)
z[rng_nc.permutation(n)[:2000]] = 1
took = np.where(z == 1, rng_nc.random(n) < 0.6, False)   # controls never get it
true_complier_effect = 5.0
Y = Y0 + took * true_complier_effect

itt = Y[z == 1].mean() - Y[z == 0].mean()     # intent-to-treat: assigned groups
compliance = took[z == 1].mean()
cace = itt / compliance                        # complier average causal effect

print(f"True effect among compliers      : {true_complier_effect:+.1f} points")
print(f"Compliance rate (took | assigned): {compliance:.0%}")
print(f"Intent-to-treat (ITT) effect     : {itt:+.2f} points   (diluted toward zero)")
print(f"Recovered complier effect (CACE) : {cace:+.2f} points   (ITT / compliance)")

In [ ]:
# Self-check: ITT is diluted; dividing by compliance recovers the complier effect.
assert itt < true_complier_effect - 1.0, "ITT should be clearly below the complier effect"
assert abs(cace - true_complier_effect) < 1.0, "ITT / compliance should recover ~5"
print(f"✓ Self-check passed: ITT ({itt:+.2f}) understates the +5 complier effect, but "
      f"ITT / compliance\n  = {cace:+.2f} recovers it. ITT and CACE answer different questions; "
      f"report which one you mean.")

### Design variants: blocking, clustering, and factorials

Three design choices change what an experiment can do. Read and run these as short
demos.

- **Blocking**: grouping units by a trait, then randomizing *within* each group.
  Example: split by prior GPA, then randomize inside each GPA band. Blocking on a
  trait tied to the outcome makes the arms more alike, which *tightens* your
  interval.
- **Cluster randomization**: assigning whole groups, not individuals. Example:
  randomize entire classrooms to a curriculum. It is often the only ethical or
  practical option, but it *widens* true uncertainty, because units in a cluster
  move together and you effectively have fewer independent observations.
- **Factorial design**: testing two or more treatments at once by randomizing each
  independently. Example: randomize email (on/off) and text (on/off) together to
  read each **main effect** (one factor's average effect) and their **interaction**
  (whether the two together do more or less than the sum of their parts).

In [ ]:
# CLUSTERING: ignoring clusters makes your interval far too narrow.
# Fresh seeded generator (seed 464). Simulate many cluster-randomized trials and
# compare the TRUE spread of estimates to the SE a naive iid analysis would report.
rng_cl = np.random.default_rng(SEED)
n_clusters, per_cluster = 40, 20
N = n_clusters * per_cluster
unit_cluster = np.arange(N) // per_cluster

def one_cluster_trial():
    cluster_shift = rng_cl.normal(0, 6, n_clusters)          # shared within-cluster level
    Y0 = np.repeat(cluster_shift, per_cluster) + rng_cl.normal(50, 4, N)
    z = np.zeros(N, dtype=int)
    for c in rng_cl.permutation(n_clusters)[:n_clusters // 2]:
        z[unit_cluster == c] = 1                             # whole clusters assigned
    Y = Y0 + z * tau
    est = Y[z == 1].mean() - Y[z == 0].mean()
    naive_se = np.sqrt(Y[z == 1].var(ddof=1) / (z == 1).sum()
                       + Y[z == 0].var(ddof=1) / (z == 0).sum())
    return est, naive_se

ests, naive_ses = zip(*[one_cluster_trial() for _ in range(400)])
true_spread = np.std(ests)
avg_naive_se = np.mean(naive_ses)
print(f"True spread of the estimate across cluster-randomized trials : {true_spread:.2f} points")
print(f"Average SE a naive 'independent units' analysis would report : {avg_naive_se:.2f} points")
print(f"The naive SE is about {true_spread/avg_naive_se:.0f}x too small, so its intervals")
print("would look far more precise than the design actually warrants.")

In [ ]:
# FACTORIAL: two treatments at once, read as two main effects plus an interaction.
# Fresh seeded generator (seed 464). Built with a real interaction of +4.
rng_f = np.random.default_rng(SEED)
n = 1200
A = rng_f.integers(0, 2, n)                    # email on/off
B = rng_f.integers(0, 2, n)                    # text on/off
Y = 50 + 3.0 * A + 2.0 * B + 4.0 * (A * B) + rng_f.normal(0, 8, n)

main_A = Y[A == 1].mean() - Y[A == 0].mean()
main_B = Y[B == 1].mean() - Y[B == 0].mean()
cell_means = {f"A={a},B={b}": Y[(A == a) & (B == b)].mean() for a in (0, 1) for b in (0, 1)}
interaction = ((cell_means["A=1,B=1"] - cell_means["A=1,B=0"])
               - (cell_means["A=0,B=1"] - cell_means["A=0,B=0"]))
print("Cell means:", {k: round(v, 1) for k, v in cell_means.items()})
print(f"Main effect of email (A)  : {main_A:+.2f} points")
print(f"Main effect of text (B)   : {main_B:+.2f} points")
print(f"Interaction (A and B)     : {interaction:+.2f} points   (built-in truth: +4)")
print("The two together do MORE than the sum of their separate effects.")

### Multiplicity and heterogeneous effects

Two last traps close the lecture.

- **Multiple-outcome testing** (multiplicity): checking many outcomes and reporting
  the one that "worked." Example: test 20 outcomes with no real effect, and about
  one clears the 0.05 bar by pure chance. An honest report shows all the tests it
  ran, not just the winner.
- **Heterogeneous treatment effect**: an effect that differs across subgroups
  rather than one number for everyone. Example: a reminder helps first-time voters
  a lot and habitual voters barely at all. Subgroups are worth exploring, but a
  difference found by slicing the data many ways can also be multiplicity in
  disguise, so treat a surprising subgroup effect as a hypothesis, not a finding.

In [ ]:
# MULTIPLICITY: 20 outcomes, NONE with a real effect. How many "work" by chance?
rng_m = np.random.default_rng(SEED)

def false_positives(k=20, n_per=200):
    hits = 0
    for _ in range(k):
        c = rng_m.normal(0, 1, n_per)          # NO true effect anywhere
        t = rng_m.normal(0, 1, n_per)
        se_j = np.sqrt(c.var(ddof=1) / n_per + t.var(ddof=1) / n_per)
        if abs((t.mean() - c.mean()) / se_j) > 1.96:
            hits += 1
    return hits

runs = np.array([false_positives() for _ in range(200)])
print(f"Testing 20 null outcomes per experiment, over 200 experiments:")
print(f"  average number that reach p < 0.05 by chance : {runs.mean():.2f}")
print(f"  share of experiments with at least one 'hit'  : {(runs >= 1).mean():.0%}")
print("So finding ONE significant outcome among many proves almost nothing on its own.")

In [ ]:
# HETEROGENEITY: the real canvass effect is not one number across wards.
ward_effect = (foos.groupby(["ward", "treat"])["marked_register_2014"].mean()
               .unstack())
ward_effect["effect_pp"] = (ward_effect[1] - ward_effect[0]) * 100
ward_effect = ward_effect.sort_values("effect_pp")
print("Contact effect on turnout by ward (percentage points):")
print(ward_effect["effect_pp"].round(1).to_string())
print(f"\nWard-level effects range from {ward_effect['effect_pp'].min():.1f} to "
      f"{ward_effect['effect_pp'].max():.1f} points around the overall {diff*100:+.1f}.")
print("Some of this spread is real variation; some is just small ward samples. Treat a")
print("striking subgroup as a question to test next, not a result to report.")

> **A question that often comes up here:** *"A study reports the one outcome out of
> twenty that reached significance. Isn't one hit still something?"* On its own, no.
> The multiplicity cell just showed that testing 20 dead outcomes yields about one
> false positive most of the time. If a report shows you only the winner and hides
> the other nineteen, you cannot tell a real effect from a lucky draw. The honest
> move is to pre-register the primary outcome, or to report every test you ran and
> adjust for how many you did.

### 📝 Practice: match the broken experiment to the threat

*(Human-first: match all four yourself before any AI. Naming the threat by eye is
the graded skill.)*

Match each broken experiment to the threat that broke it (**attrition /
noncompliance / spillover / multiple-outcome fishing**, each used once). For the
one you mark **noncompliance**, also name the group your surviving estimate
actually describes. Answer in the cell that follows.

- **A.** A weight-loss trial measures 20 outcomes and headlines the single one that
  reached significance.
- **B.** In a tutoring experiment, 40 percent of those assigned a tutor never
  showed up, so the assigned-groups difference looks small.
- **C.** A jobs program is randomized by person within each town; treated people
  share job leads with untreated neighbors, and the control group's employment
  rises too.
- **D.** A depression-app study loses 45 percent of the placebo arm to follow-up,
  mostly the people who felt worst.

### YOUR ANSWER HERE:

**A (threat):**

**B (threat + the group my estimate describes):**

**C (threat):**

**D (threat):**

---

### 🧑‍⚖️ Human-Only Checkpoint

Close Gemini for this one. No AI, no notebook search. Two decisions here are yours
alone: the ethics of your experiment and the exact quantity your M8 number
describes. In the cell below, write, in your own words:

1. **Ethics.** Experiments act on real people. Name one ethical cost your design
   imposes (a burden, a risk, withholding something valuable from the control arm)
   and how you would minimize or justify it. If your project cannot ethically
   randomize, say so, and name the observational fallback.
2. **The quantity, stated twice.** First, the exact effect your M8 estimate
   describes, worded to name the units and the population it speaks for (for
   example, "the effect among compliers," or "the intent-to-treat effect for
   everyone assigned"). Second, the overclaim you are forbidding: the tempting
   sentence that would quietly widen your estimate to units or a population it does
   not cover.

If you cannot yet name the quantity honestly, that is a finding, not a failure. It
tells you which threat you still have to handle in M8.

### YOUR ANSWER HERE:

**1. One ethical cost of my design, and how I minimize or justify it:**

**2a. The exact quantity my M8 estimate describes (units + population):**

**2b. The overclaim I forbid:**

---

### 🎯 Project Transfer: the spine of your M8 minimum viable analysis

*(Human-first: draft every piece yourself. AI may critique it after, but the
analysis you pitch at the studio has to be reasoned by you.)*

This is where the week becomes your **M8 minimum viable analysis**, the first real
estimate on your own question. In the cell below, draft its spine:

1. **The potential outcomes.** For one unit in your study, write Y(1) and Y(0) in
   plain words, and mark which one your data will actually observe.
2. **The estimate and its interval.** Name the number your M8 will report (a
   difference in means, or your pathway's equivalent) and how you will attach
   uncertainty (a randomization-inference or standard interval).
3. **Power.** State whether your design is realistically powered for the effect you
   expect, and if not, what that means for how you word your finding.
4. **The nearest threat.** Name the single threat closest to your design
   (attrition, noncompliance, spillover, multiplicity, or confounding if you are
   not randomizing) and how your analysis handles or bounds it.
5. **The quantity sentence.** One sentence stating exactly what your estimate means
   and for whom, plus the overclaim you forbid.

These five pieces are what you pitch in 60 seconds at Friday's studio. Write them so
a skeptic who has never seen your project could tell what your number does and does
not establish.

### YOUR ANSWER HERE:

**1. Potential outcomes for one unit (Y(1) / Y(0) / which is observed):**

**2. The estimate my M8 reports, and how I attach an interval:**

**3. Power (is my design realistically powered? if not, what that means):**

**4. My nearest threat, and how my analysis handles or bounds it:**

**5. My quantity sentence (what it means, for whom) and the overclaim I forbid:**

---

### 📒 AI Research Ledger

Log every AI use from this notebook in the ledger. One worked row is filled in as a
model, and it models the discipline this week teaches: **you named the quantity,
the AI pressure-tested it.** This notebook offers four prompts, one live per lecture
and two homework-depth, so your ledger carries a row for each one you actually ran.
The ledger is a graded habit, not paperwork: it is how you show your work was
verified.

| Task delegated | Tool used | Prompt | Output summary | Decision | Verification method | Remaining concern | Responsible researcher |
|---|---|---|---|---|---|---|---|
| Name which group and direction the attrition estimate describes | Gemini | "State precisely which population the survivors-only estimate now describes, and the direction of the change; name the group, not just 'biased'." | Said the estimate describes treated units who did not drop out and runs *above* the true effect | Accepted the group and direction after checking; used its wording in my quantity sentence | Read the printed survivors-only estimate (+8.0 vs true +4.0); confirmed the upward direction myself | Real studies will not hand me the true effect to check the direction against | *(your name)* |
|  |  |  |  |  |  |  |  |
|  |  |  |  |  |  |  |  |

---

### 🛡️ Exit Defense

Defense #08 — write, in your own words:

1. **The claim I can defend:** one bounded sentence, from today's experiment or your
   own project, that you would put your name on.
2. **Its boundary:** what this evidence does NOT establish. Name the compass
   position (causal reasoning) and whether random assignment, plus a clean run
   without attrition, noncompliance, or spillover, actually licenses the causal
   reading here.
3. **My uncertainty and limitations:** one sentence naming your interval, your
   power, or the threat nearest your design.
4. **AI check:** what you delegated to Gemini, whether you **accepted, changed, or
   rejected** what it returned, and the specific check that decided it (a number you
   recomputed, a direction you confirmed, an assumption you forced it to state).

### YOUR ANSWER HERE:

**1. The claim I can defend:**

**2. Its boundary (compass position + whether assignment and a clean run license the causal reading):**

**3. My uncertainty and limitations:**

**4. AI check (what I delegated, how I verified):**

---

## 4. Wrap-Up

Across two lectures you built an experiment from the ground up. You wrote the
**potential outcomes** Y(1) and Y(0), met the fundamental problem that hides one of
them, and saw how **random assignment** rescues the *average* effect by making one
group a valid stand-in for the other's missing world. You analyzed a real
get-out-the-vote experiment, estimated a **+3.4 point** effect as a difference in
means, and put a **randomization-inference** interval around it that excluded zero.
You read a **power** curve and saw that a small study can miss a real effect
entirely. Then you watched three real-world problems bend a clean estimate:
attrition pushed it up, spillover pushed it down, and noncompliance diluted it
until you divided by compliance to recover the complier effect.

> **"Random assignment earns you the word *because* at the moment of assignment.
> Attrition, noncompliance, and spillover can still change which units and which
> quantity your number describes, so name the group before you name the effect."**

Next week the course turns from *finding* evidence to *attacking* it. You have your
first real estimate, so the natural next move is to try to break it: robustness,
sensitivity, and the honest line between them and p-hacking. Your M8 minimum viable
analysis is due at Friday's studio, where you pitch it in 60 seconds and the M9
poster draft opens. Bring your estimate, its interval, and the one threat you most
fear. They are about to meet a room of skeptics. This notebook companions **RDSS
ch. 18, 'Experimental: causal.'**

---

## 5. Sources & Provenance

**Provenance of this notebook:**
- *RDSS ch. 18 'Experimental: causal' + declaration_18.1 (two-arm trial) | potential outcomes, random assignment, ATE, randomization inference, blocking/cluster/factorial designs, and the complications (attrition, noncompliance, spillover) | translated (R→Python) and adapted, concept-level, for an honors non-quant audience*
- *foos_etal.csv | rdss package data | Foos et al. UK get-out-the-vote field experiment replication; analyzed as an unweighted difference in means with a seeded randomization-inference interval (the study's own analysis is weighted) | adapted (classroom-simple version, caveat taught explicitly)*
- *fresh | seeded simulations: a toy potential-outcomes world, a two-arm power curve, and attrition / spillover / noncompliance / clustering / factorial / multiplicity demonstrations (seed 464) | newly-constructed-from-source-concept*

**Dataset attribution:** Dataset from the `rdss` package (Blair, Coppock &
Humphreys, MIT License), companion to *Research Design in the Social Sciences*
(2023).

**Readings:**
- Blair, G., Coppock, A., & Humphreys, M. (2023). *Research Design in the Social
  Sciences*, ch. 18 'Experimental: causal'. Free online:
  [book.declaredesign.org](https://book.declaredesign.org/).

---

<center>

Thank you!

</center>